# Exploring Chunking Strategies and Vector Databases

In this notebook, we explore different chunking strategies for RAG

Chunking is the process of splitting a document into smaller pieces (chunks). Each chunk can then be indexed, embedded, and retrieved. A good chunking strategy usually improves Retrieval Augmented Generation (RAG) systems

## Recommended Hardware

This notebook can run on the following hardware or remote resources

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/rag/1.rag-chunking.ipynb)  

## Software Environment

Install ROCm on your system

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Explore why chunking matters in RAG
- Apply and compare chunking strategies (simple, recursive, semantic, and LLM-based)
- Build a vector database from chunked documents
- Run similarity search and inspect how chunking affects retrieval quality

### Install Dependencies

Install the package dependencies needed for this notebook.

First, get the `aup_config.py` script locally with the package dependencies.

In [ ]:
!wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/ai-agents/rag/aup_config.py

Install the dependencies. This step may take a few minutes and only needs to be done once.

In [ ]:
from aup_config import aup_setup
aup_setup()

## Let's Chunk

### Import Libraries

In [ ]:
import os
import requests
import re

import langchain_core
from langchain_text_splitters import TokenTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS

### Download Document

We are going to download a copy of the [Vitis HLS User Guide](https://docs.amd.com/r/en-US/ug1399-vitis-hls) as content for the chunking experiments.

In [ ]:
base_url = 'https://docs.amd.com/api/khub/maps/JQtJoZLV908LbR5xokDqLw/attachments/9sLLuMlUume6oVQ1~HLSdg-JQtJoZLV908LbR5xokDqLw/content?download=true&Ft-Calling-App=ft%2Fturnkey-portal&Ft-Calling-App-Version=5.2.44'
download_dir = 'data_hls'
pdf_filename = 'vitis_hls_ug.pdf'

os.makedirs(download_dir, exist_ok=True)
if not os.path.isfile(os.path.join(download_dir, pdf_filename)):
    response = requests.get(base_url, stream=True)
    if response.status_code == 200:
        pdf_path = os.path.join(download_dir, pdf_filename)
        with open(pdf_path, 'wb') as file:
            file.write(response.content)

You will see that we get one document per PDF page because `mode` is set to `page` (default). If `mode` is set to `single`, we get one document containing all PDF content

In [ ]:
loader = PyPDFLoader(os.path.join(download_dir, pdf_filename), mode="page")
pdf_doc = loader.load()

Grab a content-dense section of the document

In [ ]:
docs = pdf_doc[23:38].copy()
len(docs)

Clean up headers and footers so we only process relevant content

In [ ]:
for doc in docs:
    doc.page_content = doc.page_content.replace('\nSend Feedback', '')
    doc.page_content = re.sub(r'^Vitis HLS User Guide\s*.*$', '',doc.page_content, flags=re.MULTILINE)
    doc.page_content = re.sub(r'^UG1399\s*.*$', '',doc.page_content, flags=re.MULTILINE)

Combine all page content into a single long string

In [ ]:
text = ""
for d in docs:
    text += d.page_content

print(f'{len(text)} character in the text')
docs_merged = [langchain_core.documents.base.Document(text)]

## Chunking Strategies

In this section, we compare four chunking strategies on the same text. We keep `chunk_size` and `chunk_overlap` fixed so the comparison is fair

For each strategy, the code follows the same flow: create a splitter, split `docs_merged`, then print basic statistics (number of chunks, total characters, and average chunk length)

In [ ]:
chunk_size = 1024
chunk_overlap = 128

First, we define a helper function that summarizes chunk outputs

`chunking_stats` takes a list of chunks and returns: total chunks, total characters, and average chunk length. This makes it easy to compare splitters using the same metrics

In [ ]:
def chunking_stats(chunks: list[langchain_core.documents.base.Document]):
    total_documents = len(chunks)
    if total_documents < 1:
        return 0, 0, 0
    total_length = sum(len(chunk.page_content) for chunk in chunks)
    avg_length = total_length / total_documents
    return total_documents, total_length, avg_length

### Simple Chunking

Simple chunking splits text into fixed-size token windows with overlap. It is fast and predictable, but it may cut ideas in the middle.

In the next code cell, we create a `TokenTextSplitter`, split `docs_merged`, and print chunk statistics for this baseline approach.

In [ ]:
text_splitter_fixed = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
documents_fixed_split = text_splitter_fixed.split_documents(docs_merged)
res = chunking_stats(documents_fixed_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### Recursive Chunking

Recursive chunking tries larger boundaries first (for example paragraphs), then falls back to smaller boundaries only when needed. This usually keeps text units more natural

In the next code cell, we build a `RecursiveCharacterTextSplitter`, split the same input text, and print the same statistics for side-by-side comparison.

In [ ]:
text_splitter_recursive = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
documents_recursive_split = text_splitter_recursive.split_documents(docs_merged)
res = chunking_stats(documents_recursive_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### Semantic Chunking

Semantic chunking uses embeddings to split where meaning changes, instead of splitting only by size. This can improve retrieval quality, but chunk sizes are less uniform.

`breakpoint_threshold_type` controls how split points are detected from semantic distance values between neighboring text segments.
With `percentile`, the splitter cuts where distances are above a chosen percentile. Lower thresholds usually create more, smaller chunks; higher thresholds create fewer, larger chunks.
You can tune this behavior with `breakpoint_threshold_amount` to match your document style and retrieval needs.

In the next cells, we configure an embedding model, build a `SemanticChunker`, split the document, and print chunk statistics.

In [ ]:
base_url = "http://localhost:11434/v1/"
api_key = ""
embed_model = OpenAIEmbeddings(model="nomic-embed-text:v1.5", base_url=base_url, api_key=api_key, check_embedding_ctx_length=False)
semantic_splitter = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")

In [ ]:
document_semantic_split = semantic_splitter.split_documents(docs_merged)
res = chunking_stats(document_semantic_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### LLM chunking

LLM chunking asks a language model to produce self-contained chunks based on meaning and structure. It can create highly coherent chunks, but it is slower and more expensive at indexing time.

In the next cells, we configure a chat model, send a chunking prompt, parse the model response into a list of chunks, convert each chunk into a `Document`, and print summary statistics

In [ ]:
llm = ChatOpenAI(model="qwen3:8b", base_url=base_url, api_key=api_key, temperature=0)

In [ ]:
chunking_prompt = ChatPromptTemplate.from_template("""
    You are an expert processing technical documents. Your task is to split the following document into
    meaningful chunks. Follow these guidelines:

    1. Chunks should contain complete ideas or concepts, each chunk should be understandable on its own
    2. More complex sections should be in smaller chunks
    3. Chunks should not be smaller than 15 words
    4. Remove headers that bring no value to the understanding of the content, such as "Chapter 1", "Section 2.3", etc
    5. Keep related information together
    6. Do not split code snippets or tables if possible
    7. Remove all `\n`

    DOCUMENT:
    {document}

    Return ONLY a valid list of strings, where each string is a chunk.
    Format your response as:
    [
      "chunk1 text",
      "chunk2 text",
      ...
    ]
    Do not forget to open and close the list with square brackets, and to put each chunk between double quotes.
    Do not include any explanations or additional text outside the list.
    """
)

In [ ]:
chunking_chain = chunking_prompt | llm
llm_response = chunking_chain.invoke({"document": docs_merged[0].page_content})

In [ ]:
llm_split = eval(llm_response.content)

In [ ]:
documents_llm_split = [langchain_core.documents.base.Document(chunk) for chunk in llm_split]
res = chunking_stats(documents_llm_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

## Building a Vector Database

Now we create one vector database per chunking strategy. Then we run the same queries against each database to see how chunking choices affect retrieved results

In [ ]:
vectorstoredb_fixed = FAISS.from_documents(documents_fixed_split, embed_model)
vectorstoredb_recursive = FAISS.from_documents(documents_recursive_split, embed_model)
vectorstoredb_semantic = FAISS.from_documents(document_semantic_split, embed_model)
vectorstoredb_llm = FAISS.from_documents(documents_llm_split, embed_model)

Now let's query the four vector databases. Each one uses content split with a different chunking technique.

In [ ]:
query="What is one of the most important constructs in your program?"

In [ ]:
similarity_result_fixed = vectorstoredb_fixed.similarity_search(query)
similarity_result_fixed[0].page_content

In [ ]:
similarity_result_recursive = vectorstoredb_recursive.similarity_search(query)
similarity_result_recursive[0].page_content

In [ ]:
similarity_result_semantic = vectorstoredb_semantic.similarity_search(query)
similarity_result_semantic[0].page_content

In [ ]:
similarity_result_llm = vectorstoredb_llm.similarity_search(query)
similarity_result_llm[0].page_content

The number of similar results returned by the vector database is controlled by `k` in `similarity_search(query)`. The default is 4. Let's change it and compare the outputs.

In [ ]:
similarity_result_llm = vectorstoredb_llm.similarity_search(query, k=10)
for idx, sim in enumerate(similarity_result_llm):
    print(f'{idx=}: {sim.page_content}\n')

Let's run another query

In [ ]:
query = "What is a stream"

In [ ]:
similarity_result_fixed = vectorstoredb_fixed.similarity_search(query)
similarity_result_fixed[0].page_content

In [ ]:
similarity_result_recursive = vectorstoredb_recursive.similarity_search(query)
similarity_result_recursive[0].page_content

In [ ]:
similarity_result_semantic = vectorstoredb_semantic.similarity_search(query)
similarity_result_semantic[0].page_content

In [ ]:
similarity_result_llm = vectorstoredb_llm.similarity_search(query)
similarity_result_llm[0].page_content

## Exercise for the Reader

- Change `chunk_size` and `chunk_overlap`
- Explore other `breakpoint_threshold_type` in the `SemanticChunker`, e.g., `standard_deviation` or `interquartile`
- Try a different VectorDB such as [Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma) by replacing `FAISS` with `Chroma`
  - Do you note any difference in the quality of results?


## Conclusions

This notebook presented several chunking techniques used in RAG systems. We built documents with each method, ingested them into vector databases, and ran similarity search to compare result quality

---

[AMD University Program](https://www.amd.com/aup)

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT